# Streaming Engineering — Kafka + Spark> **Engineering Crash Courses** · [Web verzió](./index.html) · [Vissza a főoldalra](../index.html)Ez egy futtatható **Jupyter notebook** formátum, párhuzamosan a web-alapú kurzussal.Itt ugyanazokat a kódrészleteket tudod lokálisan, saját környezetben végigcsinálni.## Hogyan futtasd```bash# 1. Virtuális környezet (Python 3.10+)python -m venv .venv# Windows:.venv\Scripts\activate# macOS/Linux:source .venv/bin/activate# 2. Telepítsd a függőségeket (a notebook első cellája)# 3. Indítsd a Jupytertjupyter lab# vagyjupyter notebook```Minden cella saját magában értelmezhető. A `# %%` kommentek Jupytekben és VSCode-ban is a cellák határát jelölik.

## 1. KörnyezetEbben a notebookban **Kafkát szimulálunk** `confluent-kafka` Python klienssel egy lokális Docker-es Kafka ellen.Indítsd el először Kafkát Dockerrel:```bashdocker run -d --name kafka-dev \  -p 9092:9092 -p 9093:9093 \  -e KAFKA_CFG_NODE_ID=0 \  -e KAFKA_CFG_PROCESS_ROLES=controller,broker \  -e KAFKA_CFG_LISTENERS=PLAINTEXT://:9092,CONTROLLER://:9093 \  -e KAFKA_CFG_ADVERTISED_LISTENERS=PLAINTEXT://localhost:9092 \  -e KAFKA_CFG_CONTROLLER_LISTENER_NAMES=CONTROLLER \  -e KAFKA_CFG_LISTENER_SECURITY_PROTOCOL_MAP=CONTROLLER:PLAINTEXT,PLAINTEXT:PLAINTEXT \  -e KAFKA_CFG_CONTROLLER_QUORUM_VOTERS=0@localhost:9093 \  bitnami/kafka:3.7```

In [ ]:
%pip install confluent-kafka==2.4.0 --quiet

## 2. Producer — clickstream események

In [ ]:
from confluent_kafka import Producerimport json, time, randomfrom datetime import datetimeproducer = Producer({'bootstrap.servers': 'localhost:9092'})topic = 'webshop.clicks'def delivery_cb(err, msg):    if err: print(f'  ✗ {err}')products = ['laptop', 'phone', 'headphones', 'camera']for i in range(10):    event = {        'event_id':    f'evt-{i:04d}',        'user_id':     random.randint(1, 100),        'product':     random.choice(products),        'timestamp':   datetime.now().isoformat(),        'session_id':  f'session-{random.randint(1, 20)}',    }    producer.produce(        topic,        key=event['session_id'],        value=json.dumps(event),        callback=delivery_cb,    )    producer.poll(0)    time.sleep(0.1)producer.flush()print(f'✓ {10} esemény elküldve a {topic} topicra')

## 3. Consumer — olvasás + aggregáció

In [ ]:
from confluent_kafka import Consumerfrom collections import Counterimport jsonconsumer = Consumer({    'bootstrap.servers': 'localhost:9092',    'group.id': 'webshop-analytics',    'auto.offset.reset': 'earliest',})consumer.subscribe([topic])counter = Counter()start = time.time()while time.time() - start < 5:  # max 5 sec várunk    msg = consumer.poll(timeout=1.0)    if msg is None: continue    if msg.error(): print('err:', msg.error()); continue    event = json.loads(msg.value())    counter[event['product']] += 1print('Termék kattintások:')for p, c in counter.most_common():    print(f'  {p:15s} {c}')consumer.close()

## 4. Event schema + kontrakt

In [ ]:
from pydantic import BaseModel, Fieldfrom datetime import datetimeclass ClickEvent(BaseModel):    """WebShop Pro clickstream event schema v1."""    event_id:    str      = Field(pattern=r'^evt-\d+$')    user_id:     int      = Field(ge=1)    product:     str    timestamp:   datetime    session_id:  str# Minden bejövő eventre validáljukraw = '{"event_id":"evt-0042","user_id":42,"product":"laptop","timestamp":"2025-05-05T10:00:00","session_id":"session-7"}'ev = ClickEvent.model_validate_json(raw)print(ev.model_dump())

## 5. Structured Streaming koncepció (Spark)Production pipeline minta — olvass Kafkából, írj Delta Lake-be:```pythonfrom pyspark.sql import SparkSessionfrom pyspark.sql import functions as Ffrom pyspark.sql.types import StructType, StringType, IntegerType, TimestampTypeschema = (StructType()    .add('event_id', StringType())    .add('user_id', IntegerType())    .add('product', StringType())    .add('timestamp', TimestampType())    .add('session_id', StringType()))stream = (spark.readStream    .format('kafka')    .option('kafka.bootstrap.servers', 'localhost:9092')    .option('subscribe', 'webshop.clicks')    .option('startingOffsets', 'earliest')    .load()    .select(F.from_json(F.col('value').cast('string'), schema).alias('e'))    .select('e.*'))query = (stream    .withWatermark('timestamp', '10 minutes')    .groupBy(F.window('timestamp', '1 minute'), 'product')    .count()    .writeStream    .format('delta')    .outputMode('append')    .option('checkpointLocation', 'chk/clicks_by_minute')    .start('lake/gold/clicks_by_minute'))query.awaitTermination()```

## Következő lépések- Térj vissza a [web-alapú kurzushoz](streaming-engineering/index.html) a teljes anyagért, diagramokért és kvízekért.- Kapcsolódó források és videók a kurzusoldal alján találhatók a "További tanulás" szekcióban.- Ha elakadsz: [GitHub Issues](https://github.com/lugosidomotor/engineering_crash_courses/issues)---*Engineering Crash Courses · MIT licenc · Magyar Data & AI Engineering kurzusok*